This notebook attempts to showcase the creation of a suitable list of largest mids (machine identifier) that are contained in both audio datasets Audioset and FSD50k. This list of mids is needed to produce a subset of needed segment files (less files needed means less conversions, I/O, and storage) and to produce accurate target labels for each file. 

In [2]:
import pandas as pd
import utils.utility_funcs as my_utils

In [3]:
# Read in the metafiles of both fsd50k and audioset

# Change column names to be more cohesive with audioset
fsd50k_class_vocab = pd.read_csv(r'C:\Users\mp431591\Documents\FSD50k_label_files\vocabulary.csv',
                                 header=None, names=["display_name", "mids"])

fsd50k_mids = set(fsd50k_class_vocab["mids"])

# Same column name change as for fsd50k
audioset_mid_to_label_mapping = pd.read_csv(r'C:\Users\mp431591\Documents\audioset_label_files\mid_to_display_name.tsv',
                                     sep="\t", header=None, names=["mids", "display_name"])
audioset_mid_to_label_mapping.head()

,mids,display_name
0,/g/11b630rrvh,Kettle whistle
1,/g/122z_qxw,Firecracker
2,/m/01280g,Wild animals
3,/m/012f08,Motor vehicle (road)
4,/m/012n7d,Ambulance (siren)


In [4]:
# Read in audiosets strongly annotated training split meta file to quantify largest classes.
# While the evaluation splits information could also be used, the training split
# is larger and is most likely good enough.

# Audioset train labels
audioset_train_classes = pd.read_csv(r'C:\Users\mp431591\Documents\audioset_label_files\audioset_train_strong.tsv',
                                     sep="\t")
audioset_train_classes.head()

,segment_id,start_time_seconds,end_time_seconds,label
0,b0RFKhbpFJA_30000,0.000,10.000,/m/03m9d0z
1,b0RFKhbpFJA_30000,4.753,5.720,/m/05zppz
2,b0RFKhbpFJA_30000,0.000,10.000,/m/07pjwq1
3,b0RFKhbpFJA_30000,6.899,7.010,/m/07qjznt
4,b0RFKhbpFJA_30000,8.534,9.156,/t/dd00092


In [5]:
# Drop temporal information since we're aiming for weak labels
audioset_train_classes = audioset_train_classes.loc[:, ['segment_id', 'label']]
audioset_train_classes.head(5)

,segment_id,label
0,b0RFKhbpFJA_30000,/m/03m9d0z
1,b0RFKhbpFJA_30000,/m/05zppz
2,b0RFKhbpFJA_30000,/m/07pjwq1
3,b0RFKhbpFJA_30000,/m/07qjznt
4,b0RFKhbpFJA_30000,/t/dd00092


In [6]:
# Rename labels column to mids column for more accurate meaning and better coherence with FSD50k
audioset_train_classes.rename(columns={'label': 'mids'}, inplace=True)
audioset_train_classes.info()
# These steps are found in a utlity function as well

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 934821 entries, 0 to 934820
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  934821 non-null  object
 1   mids        934821 non-null  object
dtypes: object(2)
memory usage: 14.3+ MB


In [7]:
# Drop duplicates in case any 10 sec clip has multiple instances of the same sound
# A quick manual look confirms these exist for at least audioset train
audioset_train_no_dups = audioset_train_classes.drop_duplicates()
audioset_train_no_dups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 432296 entries, 0 to 934813
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  432296 non-null  object
 1   mids        432296 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [8]:
# Drop rows where the mid is not found in the fsd50k mids.
rows_w_good_mid = audioset_train_no_dups.mids.isin(fsd50k_mids)
rows_w_good_mid
audioset_good_rows = audioset_train_no_dups[rows_w_good_mid]
audioset_good_rows.info()

<class 'pandas.core.frame.DataFrame'>
Index: 277487 entries, 0 to 934813
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  277487 non-null  object
 1   mids        277487 non-null  object
dtypes: object(2)
memory usage: 6.4+ MB


In [9]:
# Count mid frequencies
audioset_train_labels_by_mid_count = (audioset_good_rows.value_counts() # add a value to unique rows (in practice should be 1)
                     .groupby(by='mids') # organize by mid
                     .sum() # Sum the mid specific value i.e, in how many segments does the mid appear
                     .sort_values(ascending=False))
audioset_train_labels_by_mid_count[0:50]

mids
/m/05zppz     35107
/m/04rlf      27135
/t/dd00077    26219
/m/07qjznt    18705
/m/0lyf6      16523
/m/02zsn      14240
/m/03m9d0z    10191
/m/09l8g       8997
/m/01h8n0      7282
/m/01j3sz      6846
/m/07qcpgn     5316
/m/020bb7      4838
/m/0ytgt       4247
/t/dd00003     4232
/m/07q2z82     3757
/m/07p6fty     3467
/m/07pbtc8     3261
/m/03qtwd      3022
/m/07pggtn     2774
/m/0838f       2706
/t/dd00004     2396
/m/09x0r       2341
/m/0l15bq      2225
/m/012f08      2189
/m/0k4j        1879
/m/07jdr       1675
/m/0jbk        1668
/m/05tny_      1636
/m/032s66      1288
/m/07qnq_y     1283
/m/053hz1      1179
/m/0bt9lr      1164
/m/0912c9      1028
/m/0395lw      1017
/m/0brhx       1001
/m/07r660_      986
/m/07rrlb6      958
/m/01b_21       942
/t/dd00112      941
/m/07s0dtb      895
/m/03qc9zr      842
/m/03vt0        824
/m/014zdl       822
/m/07r04        775
/m/028ght       773
/m/02rtxlg      753
/m/015lz1       727
/m/09xqv        716
/m/0btp2        701
/m/07pp_mv     

In [32]:
# Get the top 50 - or any other subset - mids that can be used to narrow down the necessary files.
top_50_mids = audioset_train_labels_by_mid_count[0:50]
top_50_mids = set(top_50_mids.keys())

top_30_mids = audioset_train_labels_by_mid_count[0:30]
top_30_mids = set(top_30_mids.keys())

for idx, count in enumerate(audioset_train_labels_by_mid_count):
    #print(my_utils.audioset_mid_to_display_name(audioset_mid_to_label_mapping, mid))
    mid = audioset_train_labels_by_mid_count.index[idx]
    count = audioset_train_labels_by_mid_count.iloc[idx]
    readable_name = my_utils.audioset_mid_to_display_name(audioset_mid_to_label_mapping, mid)
    print(f"Label/mid: {readable_name} : {mid} and its count: {count}")

Label/mid: Male speech, man speaking : /m/05zppz and its count: 35107
Label/mid: Music : /m/04rlf and its count: 27135
Label/mid: Mechanisms : /t/dd00077 and its count: 26219
Label/mid: Tick : /m/07qjznt and its count: 18705
Label/mid: Breathing : /m/0lyf6 and its count: 16523
Label/mid: Female speech, woman speaking : /m/02zsn and its count: 14240
Label/mid: Wind : /m/03m9d0z and its count: 10191
Label/mid: Human voice : /m/09l8g and its count: 8997
Label/mid: Conversation : /m/01h8n0 and its count: 7282
Label/mid: Laughter : /m/01j3sz and its count: 6846
Label/mid: Tap : /m/07qcpgn and its count: 5316
Label/mid: Bird vocalization, bird call, bird song : /m/020bb7 and its count: 4838
Label/mid: Child speech, kid speaking : /m/0ytgt and its count: 4247
Label/mid: Male singing : /t/dd00003 and its count: 4232
Label/mid: Accelerating, revving, vroom : /m/07q2z82 and its count: 3757
Label/mid: Shout : /m/07p6fty and its count: 3467
Label/mid: Walk, footsteps : /m/07pbtc8 and its count: 32

In [28]:
audioset_train_labels_by_mid_count.index[0]

'/m/05zppz'

In [ ]:
# Save the object as a pickle object for further downstream use.
from utils.utility_funcs import pickle_save
pickle_save("mid_objects/top_50_mids.pckl", top_50_mids)

In [ ]:
pickle_save("mid_objects/top_30_mids.pckl", top_30_mids)